# RAG Research Assistant — Hybrid Retrieval, Reranking & Grounded Generation

This case study builds and evaluates a Retrieval-Augmented Generation system over a curated corpus of foundational papers on Transformers, information retrieval, and RAG.

The project is intentionally evaluated as a **retrieval system first** and a generation system second. Instead of judging the pipeline only from a few chatbot responses, the retrieval stack is compared quantitatively before the final architecture is locked.

```text
arXiv PDFs
    ↓
PyMuPDF parsing + metadata
    ↓
token-aware chunks
    ↓
┌───────────────────────┬───────────────────────┐
│ BGE dense embeddings  │ BM25 sparse retrieval │
└───────────┬───────────┴───────────┬───────────┘
            └────── Qdrant + RRF ───┘
                         ↓
               Cross-encoder reranker
                         ↓
                  Context expansion
                         ↓
                 GPT-5.6 Luna
                         ↓
             Grounded answer + citations
```

### Final configuration

- **Dense embeddings:** `BAAI/bge-small-en-v1.5`
- **Sparse retrieval:** `Qdrant/bm25`
- **Vector database:** Qdrant local mode
- **Fusion:** Reciprocal Rank Fusion
- **Reranker:** `Xenova/ms-marco-MiniLM-L-6-v2`
- **Selected chunk size:** 700 tokens
- **Hybrid candidate set:** 20
- **Final reranked results:** 6
- **Context expansion:** ±1 neighboring chunk
- **Generator:** `gpt-5.6-luna`

## 1. Experimental design

The corpus contains 11 research papers covering Transformer architecture, BERT, Sentence-BERT, DPR, ColBERT, RAG, Fusion-in-Decoder, HyDE, long-context behavior, Self-RAG, and RAFT.

The evaluation is separated into three stages:

1. **Development retrieval benchmark** — used to compare dense, sparse, hybrid, reranked retrieval, and chunk sizes.
2. **Locked source-level retrieval holdout** — evaluated only after the architecture and chunk size are selected.
3. **Qualitative grounded generation and abstention checks** — used to inspect whether the retrieved chunks actually contain enough evidence to answer.

This distinction matters because retrieval labels identify expected **papers**, not exact answer-bearing chunks. A high source-level metric does not automatically guarantee sufficient final context.

## 2. Imports and configuration

In [1]:
from pathlib import Path
from collections import defaultdict
from getpass import getpass
import json
import os
import re
import time
import uuid

from typing import Dict, List, Literal, Optional
from copy import deepcopy

import fitz
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

from dotenv import load_dotenv
from tqdm.auto import tqdm
from IPython.display import display

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from fastembed import TextEmbedding, SparseTextEmbedding
from fastembed.rerank.cross_encoder import TextCrossEncoder
import tiktoken

from qdrant_client import QdrantClient, models

from openai import OpenAI

load_dotenv()

DATA_DIR = Path("data")
PDF_DIR = DATA_DIR / "papers"
QDRANT_DIR = DATA_DIR / "qdrant"
ARTIFACT_DIR = Path("artifacts")

for path in [DATA_DIR, PDF_DIR, QDRANT_DIR, ARTIFACT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DENSE_MODEL = "BAAI/bge-small-en-v1.5"
SPARSE_MODEL = "Qdrant/bm25"
RERANK_MODEL = "Xenova/ms-marco-MiniLM-L-6-v2"

GENERATION_MODEL = "gpt-5.6-luna"
EVALUATION_MODEL = "gpt-5.6-terra"

CHUNK_SIZE = 450
CHUNK_OVERLAP = 75

DENSE_TOP_K = 12
SPARSE_TOP_K = 12
HYBRID_CANDIDATE_K = 20
RERANK_TOP_K = 6

# Final query-time retrieval configuration.
# All final retrieval evaluations use these exact values.
OPERATIONAL_CANDIDATE_K = HYBRID_CANDIDATE_K  # 20
OPERATIONAL_FINAL_K = RERANK_TOP_K            # 6
OPERATIONAL_K_VALUES = (1, 3, 5, 6)

MODEL_PRICES = {
    "gpt-5.6-luna": {"input": 0.20, "output": 1.20},
    "gpt-5.6-terra": {"input": 2.00, "output": 12.00},
}

print("Configuration loaded.")

Configuration loaded.


## 3. Corpus manifest and ingestion

In [2]:
with open("papers_manifest.json", encoding="utf-8") as f:
    papers = json.load(f)

papers_df = pd.DataFrame(papers)
papers_df

,paper_id,title,arxiv_id,url
0,transformer,Attention Is All You Need,1706.03762,https://arxiv.org/pdf/1706.03762.pdf
1,bert,BERT: Pre-training of Deep Bidirectional Trans...,1810.04805,https://arxiv.org/pdf/1810.04805.pdf
2,sbert,Sentence-BERT: Sentence Embeddings using Siame...,1908.10084,https://arxiv.org/pdf/1908.10084.pdf
3,dpr,Dense Passage Retrieval for Open-Domain Questi...,2004.04906,https://arxiv.org/pdf/2004.04906.pdf
4,colbert,ColBERT: Efficient and Effective Passage Searc...,2004.12832,https://arxiv.org/pdf/2004.12832.pdf
5,rag,Retrieval-Augmented Generation for Knowledge-I...,2005.11401,https://arxiv.org/pdf/2005.11401.pdf
6,fid,Leveraging Passage Retrieval with Generative M...,2007.01282,https://arxiv.org/pdf/2007.01282.pdf
7,hyde,Precise Zero-Shot Dense Retrieval without Rele...,2212.10496,https://arxiv.org/pdf/2212.10496.pdf
8,lost_middle,Lost in the Middle: How Language Models Use Lo...,2307.03172,https://arxiv.org/pdf/2307.03172.pdf
9,self_rag,"Self-RAG: Learning to Retrieve, Generate, and ...",2310.11511,https://arxiv.org/pdf/2310.11511.pdf


In [3]:
def download_pdf(url, destination, timeout=60):
    if destination.exists():
        return destination

    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0 RAGResearchAssistant/1.0"},
        timeout=timeout,
    )
    response.raise_for_status()
    destination.write_bytes(response.content)
    return destination


for paper in tqdm(papers):
    destination = PDF_DIR / f'{paper["paper_id"]}.pdf'
    try:
        download_pdf(paper["url"], destination)
    except Exception as exc:
        print(f'Failed {paper["paper_id"]}: {exc}')

print("PDF count:", len(list(PDF_DIR.glob("*.pdf"))))

  0%|          | 0/11 [00:00<?, ?it/s]

PDF count: 11


In [4]:
def load_pdf_pages(pdf_path: str,
                   paper_metadata: Dict):

    docs_list = []

    with fitz.open(pdf_path) as pdf:
        for i,page in enumerate(pdf):
            if not (text := page.get_text().strip()):
                continue

            page_metadata = {
                **paper_metadata,
                "page": i+1
            }
            docs_list.append(
                Document(
                    page_content=text,
                    metadata = page_metadata
                )
            )
        
    return docs_list

In [5]:
page_documents = {}
total_pages = 0
for paper_metadata in papers:
    paper_id = paper_metadata["paper_id"]
    paper_path = PDF_DIR / f'{paper_id}.pdf'

    pages = load_pdf_pages(paper_path, paper_metadata)
    page_documents[paper_id] = pages

    n_pages = len(pages)
    total_pages += n_pages
    print(f"{paper_id}: {n_pages}")

print(f"Total number of pages: {total_pages}")

example_paper_id = next(iter(page_documents))
example_document = page_documents[example_paper_id][0]

print("\nExample document:")
print(example_document.page_content[:1000])

print("\nMetadata:")
print(example_document.metadata)

transformer: 15
bert: 16
sbert: 11
dpr: 13
colbert: 10
rag: 19
fid: 6
hyde: 11
lost_middle: 18
self_rag: 30
raft: 12
Total number of pages: 161

Example document:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanis

### Corpus observations

The parser produces **161 non-empty PDF pages** across the 11 papers while preserving `paper_id`, title, arXiv identifier, source URL, and page number as metadata.

Attaching metadata during ingestion is important because citations later come from retrieved metadata rather than from the language model's memory. This makes citations traceable to the indexed corpus.

## 4. Cleaning and token-aware chunking

In [6]:
def normalize_text(text: str) -> str:
    # Remove null characters that may appear during PDF extraction
    text = text.replace("\x00", "")

    # Normalize spaces and tabs, but preserve line breaks
    text = re.sub(r"[ \t]+", " ", text)

    # Remove whitespace around line breaks
    text = re.sub(r" *\n *", "\n", text)

    # Collapse excessive blank lines while preserving paragraphs
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# Normalize all page Documents while preserving their metadata
for paper_id, documents in page_documents.items():
    page_documents[paper_id] = [
        Document(
            page_content=normalize_text(doc.page_content),
            metadata=doc.metadata.copy()
        )
        for doc in documents
    ]


# Inspect an example after normalization
example_paper_id = next(iter(page_documents))
example_document = page_documents[example_paper_id][0]

print("Normalized document:")
print(example_document.page_content[:1000])

print("\nMetadata:")
print(example_document.metadata)

Normalized document:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolu

In [7]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    add_start_index=True,
)


chunk_documents = {}
for paper_id, pages in page_documents.items():

    # Split all pages from this paper.
    # Existing metadata is preserved and start_index is added.
    paper_chunks = text_splitter.split_documents(pages)

    for paper_chunk_index, chunk in enumerate(paper_chunks):

        chunk.metadata["paper_chunk_index"] = paper_chunk_index

        # Deterministic ID:
        # same paper + page + start position -> same chunk_id
        chunk.metadata["chunk_id"] = (
            f"{paper_id}"
            f"_p{chunk.metadata['page']}"
            f"_s{chunk.metadata['start_index']}"
        )

    chunk_documents[paper_id] = paper_chunks



# Correct Chunk IDs (start index = -1)
for paper_id, chunks in chunk_documents.items():

    # Map: page number -> original page Document
    pages_by_number = {
        doc.metadata["page"]: doc
        for doc in page_documents[paper_id]
    }

    for chunk in chunks:

        if chunk.metadata["start_index"] == -1:

            page_number = chunk.metadata["page"]

            page_text = pages_by_number[
                page_number
            ].page_content

            start_index = page_text.find(
                chunk.page_content
            )

            if start_index == -1:
                raise ValueError(
                    f"Could not locate chunk in "
                    f"{paper_id}, page {page_number}"
                )

            chunk.metadata["start_index"] = start_index

            # Rebuild deterministic chunk ID
            chunk.metadata["chunk_id"] = (
                f"{paper_id}"
                f"_p{page_number}"
                f"_s{start_index}"
            )

In [8]:
encoder = tiktoken.get_encoding("cl100k_base")

total_chunks = 0
total_tokens = 0
for paper, chunks in chunk_documents.items():

    n_chunks = len(chunks)
    total_chunks+=n_chunks

    chunk_sizes = []
    for chunk in chunks:
        n_tokens = len(encoder.encode(chunk.page_content))
        chunk_sizes.append(n_tokens)
        total_tokens+=n_tokens

    avg_tokens_per_chunk = np.mean(chunk_sizes)

    print(f"Paper: {paper} | ",
          f"N. Chunks: {n_chunks} | ",
          f"Avg. Tokens per Chunk: {avg_tokens_per_chunk:.2f}")

print(f"Total Number of Chunks: {total_chunks} | Total Number of Tokens: {total_tokens}")


Paper: transformer |  N. Chunks: 32 |  Avg. Tokens per Chunk: 351.75
Paper: bert |  N. Chunks: 54 |  Avg. Tokens per Chunk: 377.54
Paper: sbert |  N. Chunks: 38 |  Avg. Tokens per Chunk: 369.39
Paper: dpr |  N. Chunks: 45 |  Avg. Tokens per Chunk: 379.49
Paper: colbert |  N. Chunks: 49 |  Avg. Tokens per Chunk: 385.63
Paper: rag |  N. Chunks: 57 |  Avg. Tokens per Chunk: 366.11
Paper: fid |  N. Chunks: 19 |  Avg. Tokens per Chunk: 381.74
Paper: hyde |  N. Chunks: 38 |  Avg. Tokens per Chunk: 365.97
Paper: lost_middle |  N. Chunks: 53 |  Avg. Tokens per Chunk: 374.40
Paper: self_rag |  N. Chunks: 81 |  Avg. Tokens per Chunk: 370.86
Paper: raft |  N. Chunks: 34 |  Avg. Tokens per Chunk: 356.21
Total Number of Chunks: 500 | Total Number of Tokens: 185675


In [9]:
chunks = [
    chunk
    for paper_chunks in chunk_documents.values()
    for chunk in paper_chunks
]


The initial 450-token configuration produces **500 chunks** and approximately **185k indexed tokens**.

Chunking is treated as a retrieval hyperparameter rather than a fixed preprocessing choice. Smaller chunks can improve local specificity but increase index size and fragment context; larger chunks preserve more surrounding information but may dilute the query-relevant span. A later ablation evaluates this tradeoff under the exact final retrieval budget.

## 5. Local retrieval models and Qdrant index

In [10]:
dense_encoder = TextEmbedding(model_name=DENSE_MODEL)
sparse_encoder = SparseTextEmbedding(model_name=SPARSE_MODEL)
reranker = TextCrossEncoder(model_name=RERANK_MODEL)

print("Local models loaded.")

Local models loaded.


In [11]:
COLLECTION_NAME = "rag_research_papers"

if (
    "client" not in globals()
    or client is None
    or not isinstance(client, QdrantClient)
):
    client = QdrantClient(path=str(QDRANT_DIR))

In [12]:
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(
        collection_name=COLLECTION_NAME
    )


client.create_collection(
    collection_name=COLLECTION_NAME,

    vectors_config={
        "dense": models.VectorParams(
            size=384,
            distance=models.Distance.COSINE,
        )
    },

    sparse_vectors_config={
        "sparse": models.SparseVectorParams(
            modifier=models.Modifier.IDF,
        )
    },
)

True

In [13]:
def to_qdrant_sparse(sparse_embedding):
    return models.SparseVector(
        indices=sparse_embedding.indices.tolist(),
        values=sparse_embedding.values.tolist(),
    )

In [14]:
texts = [
    chunk.page_content
    for chunk in chunks
]


dense_vectors = list(
    dense_encoder.passage_embed(texts)
)


sparse_vectors = list(
    sparse_encoder.passage_embed(texts)
)

points = []
for i, (chunk, dense_vector, sparse_vector) in enumerate(
    zip(chunks, dense_vectors, sparse_vectors)
):

    points.append(
        models.PointStruct(
            id=i,

            vector={
                "dense": dense_vector.tolist(),
                "sparse": to_qdrant_sparse(
                    sparse_vector
                ),
            },

            payload={
                "text": chunk.page_content,
                **chunk.metadata,
            },
        )
    )

In [15]:
client.upload_points(
    collection_name=COLLECTION_NAME,
    points=points,
    batch_size=64,
)

In [16]:
count = client.count(
    collection_name=COLLECTION_NAME,
    exact=True,
)

print(
    f"Uploaded {count.count} points "
    f"to '{COLLECTION_NAME}'."
)

Uploaded 500 points to 'rag_research_papers'.


## 6. Dense, sparse, and hybrid retrieval

In [17]:
def dense_retrieve(query, k=DENSE_TOP_K,
                   collection_name = COLLECTION_NAME):

    query_vector = next(
        dense_encoder.query_embed([query])
    )

    result = client.query_points(
        collection_name=collection_name,
        query=query_vector.tolist(),
        using="dense",
        limit=k,
        with_payload=True,
        with_vectors=False,
    )

    return result.points

In [18]:
def sparse_retrieve(query, k=SPARSE_TOP_K,
                   collection_name = COLLECTION_NAME):

    sparse_query = to_qdrant_sparse(next(
        sparse_encoder.query_embed([query])
    ))


    result = client.query_points(
        collection_name=collection_name,
        query=sparse_query,
        using="sparse",
        limit=k,
        with_payload=True,
        with_vectors=False,
    )

    return result.points

In [19]:
def hybrid_retrieve(
    query,
    k=10,
    dense_k=DENSE_TOP_K,
    sparse_k=SPARSE_TOP_K,
    collection_name=COLLECTION_NAME,
):
    # Dense representation of the query
    dense_query = next(
        dense_encoder.query_embed([query])
    )

    # Sparse/BM25 representation of the same query
    sparse_query = to_qdrant_sparse(
        next(
            sparse_encoder.query_embed([query])
        )
    )

    result = client.query_points(
        collection_name=collection_name,

        prefetch=[
            # Dense candidate retrieval
            models.Prefetch(
                query=dense_query.tolist(),
                using="dense",
                limit=dense_k,
            ),

            # Sparse candidate retrieval
            models.Prefetch(
                query=sparse_query,
                using="sparse",
                limit=sparse_k,
            ),
        ],

        # Fuse both rankings using RRF
        query=models.RrfQuery(
            rrf=models.Rrf()
        ),

        limit=k,
        with_payload=True,
        with_vectors=False,
    )

    return result.points

In [20]:
with open(
    "evaluation/retrieval_benchmark.json",
    encoding="utf-8",
) as f:
    retrieval_benchmark = json.load(f)

pd.DataFrame(retrieval_benchmark).head()

,query,expected_source_ids,type
0,What mechanism lets the Transformer model depe...,[transformer],single_source
1,What pretraining objectives are introduced in ...,[bert],single_source
2,Why is vanilla BERT inefficient for large-scal...,[sbert],single_source
3,How are the question encoder and passage encod...,[dpr],single_source
4,What is late interaction in ColBERT?,[colbert],single_source


In [21]:
def retrieved_source_ids(points, k=None):
    if k is not None:
        points = points[:k]

    return [
        point.payload["paper_id"] for point in points
    ]


def hit_at_k(retrieved_ids, expected_ids, k):
    return int(len(
        set(retrieved_ids[:k]).intersection(set(expected_ids))
    ) > 0)


def recall_at_k(retrieved_ids, expected_ids, k):

    if not expected_ids:
        return 0.0

    return len(
        set(retrieved_ids[:k]).
        intersection(set(expected_ids))
    )/len(set(expected_ids))


def reciprocal_rank(retrieved_ids, expected_ids):
    expected_ids = set(expected_ids)

    for rank, retrieved_id in enumerate(retrieved_ids, start=1):
        if retrieved_id in expected_ids:
            return 1 / rank

    return 0.0

In [22]:
def evaluate_retriever(
    retriever_fn,
    benchmark,
    k_values=(1, 3, 5, 10)
):

    results = []
    for entry in benchmark:

        query = entry['query']
        max_k = max(k_values)

        retriever_params = {
            'query': query,
            'k': max_k
        }

        if retriever_fn.__name__ == 'hybrid_retrieve':
            retriever_params = {
                **retriever_params,
                'dense_k': 4 * max_k,
                'sparse_k': 4 * max_k
            }

        points = retriever_fn(**retriever_params)

        expected_ids = entry['expected_source_ids']
        retrieved_ids = retrieved_source_ids(points)

        entry_results = {f'RR@{max_k}': reciprocal_rank(retrieved_ids, expected_ids)}
        for k in k_values:
            entry_results[f"Hit@{k}"] = hit_at_k(retrieved_ids, expected_ids, k)
            entry_results[f"Recall@{k}"] = recall_at_k(retrieved_ids, expected_ids, k)

        results.append(entry_results)

    return pd.DataFrame(results)

In [23]:
dense_benchmark_result = evaluate_retriever(dense_retrieve,
                   retrieval_benchmark)

sparse_benchmark_result = evaluate_retriever(sparse_retrieve,
                   retrieval_benchmark)

hybrid_benchmark_result = evaluate_retriever(hybrid_retrieve,
                   retrieval_benchmark)

In [24]:
retrieval_benchmark_result = pd.DataFrame(
    {'Dense Retrieval': dense_benchmark_result.mean(),
     'Sparse Retrieval': sparse_benchmark_result.mean(),
     'Hybrid Retrieval': hybrid_benchmark_result.mean()}
)

retrieval_benchmark_result.index = 'Mean ' + retrieval_benchmark_result.index.astype(str)
retrieval_benchmark_result

,Dense Retrieval,Sparse Retrieval,Hybrid Retrieval
Mean RR@10,0.9375,0.895833,1.000
Mean Hit@1,0.9000,0.850000,1.000
Mean Recall@1,0.7250,0.675000,0.800
Mean Hit@3,0.9500,0.900000,1.000
Mean Recall@3,0.8250,0.750000,0.850
Mean Hit@5,1.0000,0.950000,1.000
Mean Recall@5,0.8500,0.850000,0.875
Mean Hit@10,1.0000,1.000000,1.000
Mean Recall@10,0.9750,0.950000,0.950


### Dense vs. sparse retrieval

The aggregate benchmark confirms that dense and lexical retrieval fail differently.

| Metric | Dense | BM25 | Hybrid RRF |
|---|---:|---:|---:|
| MRR@10 | 0.9375 | 0.8958 | **1.0000** |
| Hit@1 | 0.900 | 0.850 | **1.000** |
| Recall@1 | 0.725 | 0.675 | **0.800** |
| Recall@5 | 0.850 | 0.850 | **0.875** |
| Recall@10 | **0.975** | 0.950 | 0.950 |

BM25 is strongest when the query contains distinctive terminology such as explicit model names. Dense retrieval is stronger on paraphrases and relational questions where exact token overlap is weaker.

RRF combines those complementary signals and places at least one expected source at rank 1 for every development query. However, the result also exposes an important limitation: **fusion improves early ranking but does not guarantee better deep or multi-source recall**. Dense retrieval alone has slightly higher `Recall@10`.

Several failures occur because multiple high-ranked chunks from one relevant paper occupy the ranking while a second required paper is pushed below the cutoff.

## 7. Cross-encoder reranking

In [25]:
def rerank_points(query, points, top_k=RERANK_TOP_K):

    texts = [point.payload["text"] for point in points]
    scores = list(reranker.rerank(query, texts))

    new_score_points = deepcopy(points)
    for i, point in enumerate(new_score_points):
        point.score = float(scores[i])

    ranked = sorted(
        zip(scores, new_score_points),
        key=lambda x: x[0],
        reverse=True
    )

    return [point for _, point in ranked[:top_k]]


def hybrid_rerank_retrieve(
    query,
    candidate_k=HYBRID_CANDIDATE_K,
    final_k=RERANK_TOP_K,
    collection_name = COLLECTION_NAME
):
    retrieved_points = hybrid_retrieve(
        query=query,
        k=candidate_k,
        dense_k=candidate_k * 4,
        sparse_k=candidate_k * 4,
        collection_name=collection_name
    )

    return rerank_points(
        query=query,
        points=retrieved_points,
        top_k=final_k
    )

In [26]:
def evaluate_reranked_retriever(
    reranked_retriever_fn,
    benchmark,
    k_values=OPERATIONAL_K_VALUES,
    candidate_k=OPERATIONAL_CANDIDATE_K,
    final_k=OPERATIONAL_FINAL_K,
):
    """
    Evaluate the reranked retriever using the SAME candidate and
    final cutoffs used by the query-time RAG pipeline.

    The labels are source-level paper IDs, so these metrics measure
    source retrieval rather than exact evidence-passage relevance.
    """
    if max(k_values) > final_k:
        raise ValueError(
            "All k_values must be <= final_k so the metric is "
            "computed on results that would actually reach the pipeline."
        )

    results = []

    for entry in benchmark:
        query = entry["query"]

        points = reranked_retriever_fn(
            query=query,
            candidate_k=candidate_k,
            final_k=final_k,
        )

        expected_ids = entry["expected_source_ids"]
        retrieved_ids = retrieved_source_ids(points)

        entry_results = {
            "query": query,
            "expected_source_ids": expected_ids,
            "retrieved_source_ids": retrieved_ids,
            f"RR@{final_k}": reciprocal_rank(
                retrieved_ids,
                expected_ids,
            ),
        }

        for k in k_values:
            entry_results[f"Hit@{k}"] = hit_at_k(
                retrieved_ids,
                expected_ids,
                k,
            )
            entry_results[f"Recall@{k}"] = recall_at_k(
                retrieved_ids,
                expected_ids,
                k,
            )

        results.append(entry_results)

    return pd.DataFrame(results)

In [27]:
def hybrid_operational_retrieve(
    query,
    k=OPERATIONAL_FINAL_K,
):
    """
    Hybrid RRF retrieval evaluated under the same first-stage
    candidate budget used by the reranked pipeline.
    """
    return hybrid_retrieve(
        query=query,
        k=k,
        dense_k=OPERATIONAL_CANDIDATE_K * 4,
        sparse_k=OPERATIONAL_CANDIDATE_K * 4,
        collection_name=COLLECTION_NAME,
    )


hybrid_operational_benchmark = evaluate_retriever(
    hybrid_operational_retrieve,
    retrieval_benchmark,
    k_values=OPERATIONAL_K_VALUES,
)

hybrid_reranked_benchmark_result = evaluate_reranked_retriever(
    hybrid_rerank_retrieve,
    retrieval_benchmark,
    k_values=OPERATIONAL_K_VALUES,
    candidate_k=OPERATIONAL_CANDIDATE_K,
    final_k=OPERATIONAL_FINAL_K,
)

In [28]:
operational_retrieval_comparison = pd.DataFrame({
    "Hybrid RRF": hybrid_operational_benchmark.mean(
        numeric_only=True
    ),
    "Hybrid + Reranker": hybrid_reranked_benchmark_result.mean(
        numeric_only=True
    ),
})

operational_retrieval_comparison

,Hybrid RRF,Hybrid + Reranker
RR@6,1.000,0.9625
Hit@1,1.000,0.9500
Recall@1,0.800,0.7750
Hit@3,1.000,0.9500
Recall@3,0.850,0.9000
Hit@5,1.000,1.0000
Recall@5,0.875,0.9250
Hit@6,1.000,1.0000
Recall@6,0.875,0.9250


### Reranking tradeoff

Under the **same operational budget used by the final pipeline** (`20` candidates → `6` final results), reranking changes the error profile rather than simply improving every metric.

| Metric | Hybrid RRF | Hybrid + reranker |
|---|---:|---:|
| MRR@6 | **1.0000** | 0.9625 |
| Hit@1 | **1.000** | 0.950 |
| Recall@3 | 0.850 | **0.900** |
| Recall@5 | 0.875 | **0.925** |
| Recall@6 | 0.875 | **0.925** |

The cross-encoder slightly hurts first-result ranking but improves later source coverage, especially for multi-source queries. Because the final generator needs enough evidence rather than only one relevant chunk at rank 1, the reranker is retained.

This also illustrates why reranking must be evaluated against the system objective: a more expensive second-stage model is not automatically better on every ranking metric.

## 8. Chunk-size ablation

In [29]:
CHUNK_SIZE_CANDIDATES = [250, 450, 700]


def build_chunks_for_size(
    chunk_size,
    chunk_overlap=CHUNK_OVERLAP,
):
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        encoding_name="cl100k_base",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        add_start_index=True,
    )

    documents = [
        doc
        for paper_docs in page_documents.values()
        for doc in paper_docs
    ]

    return splitter.split_documents(documents)


def build_temporary_collection(
    chunks,
    collection_name,
):
    if client.collection_exists(collection_name):
        client.delete_collection(
            collection_name=collection_name
        )

    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            "dense": models.VectorParams(
                size=384,
                distance=models.Distance.COSINE,
            )
        },
        sparse_vectors_config={
            "sparse": models.SparseVectorParams(
                modifier=models.Modifier.IDF,
            )
        },
    )

    texts = [
        chunk.page_content
        for chunk in chunks
    ]

    dense_vectors = list(
        dense_encoder.passage_embed(texts)
    )

    sparse_vectors = list(
        sparse_encoder.passage_embed(texts)
    )

    points = []

    for i, (chunk, dense_vector, sparse_vector) in enumerate(
        zip(chunks, dense_vectors, sparse_vectors)
    ):
        points.append(
            models.PointStruct(
                id=i,
                vector={
                    "dense": dense_vector.tolist(),
                    "sparse": to_qdrant_sparse(
                        sparse_vector
                    ),
                },
                payload={
                    "text": chunk.page_content,
                    **chunk.metadata,
                },
            )
        )

    client.upload_points(
        collection_name=collection_name,
        points=points,
        batch_size=64,
    )


def hybrid_retrieve_from_collection(
    query,
    collection_name,
    k,
    dense_k,
    sparse_k,
):
    dense_query = next(
        dense_encoder.query_embed([query])
    )

    sparse_query = to_qdrant_sparse(
        next(
            sparse_encoder.query_embed([query])
        )
    )

    result = client.query_points(
        collection_name=collection_name,
        prefetch=[
            models.Prefetch(
                query=dense_query.tolist(),
                using="dense",
                limit=dense_k,
            ),
            models.Prefetch(
                query=sparse_query,
                using="sparse",
                limit=sparse_k,
            ),
        ],
        query=models.RrfQuery(
            rrf=models.Rrf()
        ),
        limit=k,
        with_payload=True,
        with_vectors=False,
    )

    return result.points


def rerank_points_for_ablation(
    query,
    points,
    top_k,
):
    if not points:
        return []

    texts = [
        point.payload["text"]
        for point in points
    ]

    scores = list(
        reranker.rerank(
            query,
            texts,
        )
    )

    reranked_points = deepcopy(points)

    for point, score in zip(
        reranked_points,
        scores,
    ):
        point.score = float(score)

    ranked = sorted(
        zip(scores, reranked_points),
        key=lambda x: x[0],
        reverse=True,
    )

    return [
        point
        for _, point in ranked[:top_k]
    ]


def full_retrieve_from_collection(
    query,
    collection_name,
    candidate_k=OPERATIONAL_CANDIDATE_K,
    final_k=OPERATIONAL_FINAL_K,
):
    candidates = hybrid_retrieve_from_collection(
        query=query,
        collection_name=collection_name,
        k=candidate_k,
        dense_k=candidate_k * 4,
        sparse_k=candidate_k * 4,
    )

    return rerank_points_for_ablation(
        query=query,
        points=candidates,
        top_k=final_k,
    )


def evaluate_chunk_collection(
    collection_name,
    benchmark,
    k_values=OPERATIONAL_K_VALUES,
    candidate_k=OPERATIONAL_CANDIDATE_K,
    final_k=OPERATIONAL_FINAL_K,
):
    if max(k_values) > final_k:
        raise ValueError(
            "All k_values must be <= final_k."
        )

    results = []

    for entry in benchmark:
        query = entry["query"]

        points = full_retrieve_from_collection(
            query=query,
            collection_name=collection_name,
            candidate_k=candidate_k,
            final_k=final_k,
        )

        expected_ids = entry["expected_source_ids"]
        retrieved_ids = retrieved_source_ids(points)

        entry_results = {
            "query": query,
            "expected_source_ids": expected_ids,
            "retrieved_source_ids": retrieved_ids,
            f"RR@{final_k}": reciprocal_rank(
                retrieved_ids,
                expected_ids,
            ),
        }

        for k in k_values:
            entry_results[f"Hit@{k}"] = hit_at_k(
                retrieved_ids,
                expected_ids,
                k,
            )
            entry_results[f"Recall@{k}"] = recall_at_k(
                retrieved_ids,
                expected_ids,
                k,
            )

        results.append(entry_results)

    return pd.DataFrame(results)


ablation_summary = []
ablation_per_query = {}

for chunk_size in CHUNK_SIZE_CANDIDATES:
    collection_name = (
        f"rag_chunk_ablation_{chunk_size}"
    )

    print(
        f"\n{'=' * 60}\n"
        f"Chunk size: {chunk_size}\n"
        f"{'=' * 60}"
    )

    chunks_for_size = build_chunks_for_size(
        chunk_size=chunk_size
    )

    chunk_count = len(
        chunks_for_size
    )

    print(
        f"Number of chunks: {chunk_count}"
    )

    build_temporary_collection(
        chunks=chunks_for_size,
        collection_name=collection_name,
    )

    benchmark_results = evaluate_chunk_collection(
        collection_name=collection_name,
        benchmark=retrieval_benchmark,
        k_values=OPERATIONAL_K_VALUES,
        candidate_k=OPERATIONAL_CANDIDATE_K,
        final_k=OPERATIONAL_FINAL_K,
    )

    ablation_per_query[chunk_size] = (
        benchmark_results.copy()
    )

    ablation_summary.append({
        "Chunk Size": chunk_size,
        "Chunk Count": chunk_count,
        "Hit@3": benchmark_results[
            "Hit@3"
        ].mean(),
        "Recall@5": benchmark_results[
            "Recall@5"
        ].mean(),
        "Recall@6": benchmark_results[
            "Recall@6"
        ].mean(),
        "MRR@6": benchmark_results[
            "RR@6"
        ].mean(),
    })

    client.delete_collection(
        collection_name=collection_name
    )


chunk_size_benchmark = pd.DataFrame(
    ablation_summary
)

chunk_size_benchmark


Chunk size: 250
Number of chunks: 975

Chunk size: 450
Number of chunks: 500

Chunk size: 700
Number of chunks: 327


,Chunk Size,Chunk Count,Hit@3,Recall@5,Recall@6,MRR@6
0,250,975,1.00,0.900,0.925,1.0000
1,450,500,0.95,0.925,0.925,0.9625
2,700,327,1.00,0.925,0.925,1.0000


In [30]:
chunk_selection_table = (
    chunk_size_benchmark
    .sort_values(
        by=[
            "Recall@5",
            "MRR@6",
            "Chunk Count",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

optimal_chunk_size = int(
    chunk_selection_table.loc[
        0,
        "Chunk Size",
    ]
)

print("Chunk-size selection table:")
display(chunk_selection_table)

print(
    f"\nSelected chunk size: "
    f"{optimal_chunk_size}"
)

Chunk-size selection table:


,Chunk Size,Chunk Count,Hit@3,Recall@5,Recall@6,MRR@6
0,700,327,1.00,0.925,0.925,1.0000
1,450,500,0.95,0.925,0.925,0.9625
2,250,975,1.00,0.900,0.925,1.0000



Selected chunk size: 700


### Chunk-size selection

The ablation is evaluated with the **full operational pipeline**:

```text
20 hybrid candidates
→ cross-encoder reranking
→ top 6 final results
```

| Chunk size | Chunks | Hit@3 | Recall@5 | Recall@6 | MRR@6 |
|---:|---:|---:|---:|---:|---:|
| 250 | 975 | **1.00** | 0.900 | 0.925 | **1.0000** |
| 450 | 500 | 0.95 | **0.925** | 0.925 | 0.9625 |
| 700 | **327** | **1.00** | **0.925** | 0.925 | **1.0000** |

The selection rule is fixed before holdout evaluation:

1. maximize `Recall@5`;
2. among ties, maximize `MRR@6`;
3. among remaining ties, prefer fewer chunks.

This selects **700 tokens**. It preserves the strongest measured retrieval quality while reducing the index from 975 chunks at 250 tokens to 327 chunks — roughly a two-thirds reduction.

The result is specific to this corpus; it is not evidence that larger chunks are universally better.

## 9. Context expansion and grounded generation

In [31]:
def expand_context(
    points,
    neighbor_radius=1,
    collection_name=COLLECTION_NAME
):
    expanded_documents = []
    seen = set()

    for point in points:

        paper_id = point.payload["paper_id"]
        chunk_index = point.payload["paper_chunk_index"]

        start_index = max(
            0,
            chunk_index - neighbor_radius
        )

        end_index = (
            chunk_index + neighbor_radius
        )

        # Retrieve neighboring chunks directly from Qdrant
        neighbor_points, _ = client.scroll(
            collection_name=collection_name,

            scroll_filter=models.Filter(
                must=[
                    models.FieldCondition(
                        key="paper_id",
                        match=models.MatchValue(
                            value=paper_id
                        )
                    ),

                    models.FieldCondition(
                        key="paper_chunk_index",
                        range=models.Range(
                            gte=start_index,
                            lte=end_index
                        )
                    )
                ]
            ),

            limit=2 * neighbor_radius + 1,
            with_payload=True,
            with_vectors=False,
        )

        # Qdrant does not guarantee the desired chunk order here,
        # so explicitly restore paper order.
        neighbor_points = sorted(
            neighbor_points,
            key=lambda p: p.payload["paper_chunk_index"]
        )

        for neighbor in neighbor_points:

            key = (
                neighbor.payload["paper_id"],
                neighbor.payload["paper_chunk_index"]
            )

            # Prevent duplicates caused by overlapping neighborhoods
            if key in seen:
                continue

            seen.add(key)

            metadata = {
                key: value
                for key, value in neighbor.payload.items()
                if key != "text"
            }

            expanded_documents.append(
                Document(
                    page_content=neighbor.payload["text"],
                    metadata=metadata
                )
            )

    return expanded_documents

In [32]:
def format_context(documents):

    blocks = []

    for i, document in enumerate(documents, start=1):

        metadata = document.metadata

        block = (
            f"[SOURCE {i}]\n"
            f"paper_id: {metadata.get('paper_id', '')}\n"
            f"title: {metadata.get('title', '')}\n"
            f"page: {metadata.get('page', '')}\n"
            f"text: {document.page_content.strip()}"
        )

        blocks.append(block)

    return "\n\n".join(blocks)

In [33]:
ENV_PATH = Path( r".\.env")
print("File exists:", ENV_PATH.exists())
print("Path:", ENV_PATH.resolve())

load_dotenv(
    dotenv_path=ENV_PATH,
    override=True,
    encoding="utf-8-sig"
)

openai_client = OpenAI()

SYSTEM_PROMPT = '''
You are a research assistant answering questions ONLY from
the supplied context.

Rules:
1. Use only information supported by the supplied sources.
2. If evidence is insufficient, say so explicitly.
3. Cite factual claims using [SOURCE N].
4. Do not invent citations.
5. Prefer concise technical explanations.
'''.strip()

File exists: True
Path: C:\Users\avado\OneDrive\Desktop\DataEngineeringStudy\Repository\data-ai-engineering-portfolio\generative-ai\01_rag_research_assistant\.env


In [34]:
def generate_grounded_answer(
    question,
    context,
    model=GENERATION_MODEL,
):
    response = openai_client.responses.create(
        model=model,
        instructions=SYSTEM_PROMPT,
        input=(
            f"Question:\n{question}\n\n"
            f"Context:\n{context}"
        ),
    )

    return response.output_text, response.usage


def usage_cost(usage, model = GENERATION_MODEL):

    input_tokens = usage.input_tokens
    output_tokens = usage.output_tokens

    prices = MODEL_PRICES[model]

    return (
        input_tokens / 1_000_000 * prices["input"]
        + output_tokens / 1_000_000 * prices["output"]
    )

In [35]:
def answer_question(
    question,
    *,
    candidate_k=HYBRID_CANDIDATE_K,
    final_k=RERANK_TOP_K,
    neighbor_radius=1,
    collection_name = COLLECTION_NAME
):
    #
    # 1. hybrid retrieval
    # 2. cross-encoder reranking
    # 3. context expansion
    # 4. context formatting
    # 5. grounded generation
    #
    # Return answer, sources, usage and cost.

    # Step 1: Points Retrieval from DB (Re-Ranked)

    db_points = hybrid_rerank_retrieve(
        query = question,
        candidate_k = candidate_k,
        final_k = final_k,
        collection_name=collection_name
    )

    # Step 2: Context Expansion
    expanded_context_documents = expand_context(
        points = db_points,
        neighbor_radius=neighbor_radius,
        collection_name=collection_name
    )

    sources = [
        {
            "source": i,
            "paper_id": document.metadata.get("paper_id"),
            "title": document.metadata.get("title"),
            "page": document.metadata.get("page"),
            "paper_chunk_index": document.metadata.get("paper_chunk_index"),
        }
        for i, document in enumerate(
            expanded_context_documents,
            start=1
        )
    ]

    # Step 3: Expand Context
    formatted_context = format_context(
        expanded_context_documents
    )

    # Step 4: Grounded Generation + Usage Measure
    answer, usage = generate_grounded_answer(
        question = question,
        context = formatted_context
    )

    cost = usage_cost(usage)


    return answer, sources, usage, cost

Retrieval granularity and generation context are intentionally separated. The retriever ranks individual chunks, then the pipeline restores neighboring chunks around the selected results before generation.

This **small-to-big** pattern attempts to preserve retrieval precision while giving the generator enough local context to interpret the retrieved passage. Source labels are created from stored metadata, so citations are not invented by the model.

## 10. Final collection and locked retrieval holdout

In [36]:

# Create Final Collection With Optimal Chunk Size

PRODUCTION_CHUNK_SIZE = optimal_chunk_size
PRODUCTION_COLLECTION_NAME = (
    f"rag_research_papers_prod_{PRODUCTION_CHUNK_SIZE}"
)
PRODUCTION_CHUNK_OVERLAP = CHUNK_OVERLAP   # 75

production_splitter = (
    RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        encoding_name="cl100k_base",
        chunk_size=PRODUCTION_CHUNK_SIZE,
        chunk_overlap=PRODUCTION_CHUNK_OVERLAP,
        add_start_index=True,
    )
)


production_chunk_documents = {}


for paper_id, pages in page_documents.items():

    paper_chunks = production_splitter.split_documents(
        pages
    )

    # Map page number -> original normalized page
    # Used only if LangChain returns start_index = -1
    pages_by_number = {
        page.metadata["page"]: page
        for page in pages
    }

    for paper_chunk_index, chunk in enumerate(
        paper_chunks
    ):

        # Position of this chunk inside the paper.
        # Required later for neighbor context expansion.
        chunk.metadata["paper_chunk_index"] = (
            paper_chunk_index
        )

        page_number = chunk.metadata["page"]
        start_index = chunk.metadata["start_index"]

        # Correct unresolved start indices if necessary
        if start_index == -1:

            page_text = (
                pages_by_number[
                    page_number
                ].page_content
            )

            start_index = page_text.find(
                chunk.page_content
            )

            if start_index == -1:
                raise ValueError(
                    f"Could not locate chunk in "
                    f"{paper_id}, page {page_number}"
                )

            chunk.metadata["start_index"] = (
                start_index
            )

        # Deterministic chunk identifier
        chunk.metadata["chunk_id"] = (
            f"{paper_id}"
            f"_p{page_number}"
            f"_s{start_index}"
        )

    production_chunk_documents[
        paper_id
    ] = paper_chunks


production_chunks = [
    chunk
    for paper_chunks
    in production_chunk_documents.values()
    for chunk in paper_chunks
]


print(
    f"Production chunks: "
    f"{len(production_chunks)}"
)


if client.collection_exists(
    PRODUCTION_COLLECTION_NAME
):
    client.delete_collection(
        collection_name=PRODUCTION_COLLECTION_NAME
    )


client.create_collection(
    collection_name=PRODUCTION_COLLECTION_NAME,

    vectors_config={
        "dense": models.VectorParams(
            size=384,
            distance=models.Distance.COSINE,
        )
    },

    sparse_vectors_config={
        "sparse": models.SparseVectorParams(
            modifier=models.Modifier.IDF,
        )
    },
)


production_texts = [
    chunk.page_content
    for chunk in production_chunks
]


production_dense_vectors = list(
    dense_encoder.passage_embed(
        production_texts
    )
)


production_sparse_vectors = list(
    sparse_encoder.passage_embed(
        production_texts
    )
)


production_points = []


for i, (
    chunk,
    dense_vector,
    sparse_vector,
) in enumerate(
    zip(
        production_chunks,
        production_dense_vectors,
        production_sparse_vectors,
    )
):

    production_points.append(
        models.PointStruct(
            id=i,

            vector={
                "dense": dense_vector.tolist(),

                "sparse": to_qdrant_sparse(
                    sparse_vector
                ),
            },

            payload={
                "text": chunk.page_content,
                **chunk.metadata,
            },
        )
    )


client.upload_points(
    collection_name=PRODUCTION_COLLECTION_NAME,
    points=production_points,
    batch_size=64,
)


production_count = client.count(
    collection_name=PRODUCTION_COLLECTION_NAME,
    exact=True,
)


print(
    f"Production collection: "
    f"{PRODUCTION_COLLECTION_NAME}"
)

print(
    f"Uploaded points: "
    f"{production_count.count}"
)

print(
    f"Chunk size: "
    f"{PRODUCTION_CHUNK_SIZE}"
)

print(
    f"Chunk overlap: "
    f"{PRODUCTION_CHUNK_OVERLAP}"
)

Production chunks: 327
Production collection: rag_research_papers_prod_700
Uploaded points: 327
Chunk size: 700
Chunk overlap: 75


In [37]:
# ============================================================
# Final collection wrappers
# ============================================================

def production_retrieve(
    query,
    candidate_k=OPERATIONAL_CANDIDATE_K,
    final_k=OPERATIONAL_FINAL_K,
):
    return hybrid_rerank_retrieve(
        query=query,
        candidate_k=candidate_k,
        final_k=final_k,
        collection_name=PRODUCTION_COLLECTION_NAME,
    )


def production_answer(
    question,
    *,
    candidate_k=OPERATIONAL_CANDIDATE_K,
    final_k=OPERATIONAL_FINAL_K,
    neighbor_radius=1,
):
    return answer_question(
        question=question,
        candidate_k=candidate_k,
        final_k=final_k,
        neighbor_radius=neighbor_radius,
        collection_name=PRODUCTION_COLLECTION_NAME,
    )


# Re-check the development benchmark on the selected chunk size.
# These queries were used during development, so this is NOT the
# final unbiased retrieval estimate.
production_dev_benchmark = evaluate_reranked_retriever(
    production_retrieve,
    retrieval_benchmark,
    k_values=OPERATIONAL_K_VALUES,
    candidate_k=OPERATIONAL_CANDIDATE_K,
    final_k=OPERATIONAL_FINAL_K,
)

production_dev_summary = (
    production_dev_benchmark
    .mean(numeric_only=True)
)

print("Selected collection — development benchmark:")
display(production_dev_summary)

Selected collection — development benchmark:


RR@6        1.000
Hit@1       1.000
Recall@1    0.800
Hit@3       1.000
Recall@3    0.875
Hit@5       1.000
Recall@5    0.925
Hit@6       1.000
Recall@6    0.925
dtype: float64

In [38]:
final_holdout_benchmark = [
    {
        "query": (
            "Why is the attention dot product divided by the "
            "square root of the key dimension?"
        ),
        "expected_source_ids": ["transformer"],
        "type": "single_source",
    },
    {
        "query": (
            "How are token, segment, and position embeddings "
            "combined in BERT's input representation?"
        ),
        "expected_source_ids": ["bert"],
        "type": "single_source",
    },
    {
        "query": (
            "Which pooling strategies are considered when turning "
            "BERT outputs into a fixed-size sentence representation "
            "in Sentence-BERT?"
        ),
        "expected_source_ids": ["sbert"],
        "type": "single_source",
    },
    {
        "query": (
            "What kinds of negative passages are used to train "
            "Dense Passage Retrieval?"
        ),
        "expected_source_ids": ["dpr"],
        "type": "single_source",
    },
    {
        "query": (
            "How does ColBERT's MaxSim operation use query-token "
            "and document-token representations?"
        ),
        "expected_source_ids": ["colbert"],
        "type": "single_source",
    },
    {
        "query": (
            "At what stage does Fusion-in-Decoder combine information "
            "from multiple retrieved passages?"
        ),
        "expected_source_ids": ["fid"],
        "type": "single_source",
    },
    {
        "query": (
            "How is the hypothetical document generated by HyDE used "
            "to retrieve real documents?"
        ),
        "expected_source_ids": ["hyde"],
        "type": "single_source",
    },
    {
        "query": (
            "How does question-answering performance change when "
            "relevant information is placed near the beginning, "
            "middle, or end of a long context?"
        ),
        "expected_source_ids": ["lost_middle"],
        "type": "single_source",
    },
    {
        "query": (
            "How can Self-RAG decide whether retrieval is needed "
            "during generation?"
        ),
        "expected_source_ids": ["self_rag"],
        "type": "single_source",
    },
    {
        "query": (
            "What purpose do distractor documents serve during "
            "RAFT training?"
        ),
        "expected_source_ids": ["raft"],
        "type": "single_source",
    },
    {
        "query": (
            "Compare DPR and ColBERT in how they represent and score "
            "queries against documents."
        ),
        "expected_source_ids": ["dpr", "colbert"],
        "type": "multi_source",
    },
    {
        "query": (
            "How do Self-RAG and RAFT differ in the way retrieved "
            "documents are used to improve generation?"
        ),
        "expected_source_ids": ["self_rag", "raft"],
        "type": "multi_source",
    },
]

print(
    "Final holdout queries:",
    len(final_holdout_benchmark),
)

Final holdout queries: 12


In [39]:
final_holdout_results = evaluate_reranked_retriever(
    production_retrieve,
    final_holdout_benchmark,
    k_values=OPERATIONAL_K_VALUES,
    candidate_k=OPERATIONAL_CANDIDATE_K,
    final_k=OPERATIONAL_FINAL_K,
)

final_holdout_summary = (
    final_holdout_results
    .mean(numeric_only=True)
)

print("FINAL SOURCE-LEVEL RETRIEVAL HOLDOUT")
print("=" * 80)
display(final_holdout_summary)

print("\nPer-query holdout results:")
display(final_holdout_results)

FINAL SOURCE-LEVEL RETRIEVAL HOLDOUT


RR@6        1.000000
Hit@1       1.000000
Recall@1    0.916667
Hit@3       1.000000
Recall@3    0.958333
Hit@5       1.000000
Recall@5    0.958333
Hit@6       1.000000
Recall@6    0.958333
dtype: float64


Per-query holdout results:


,query,expected_source_ids,retrieved_source_ids,RR@6,Hit@1,Recall@1,Hit@3,Recall@3,Hit@5,Recall@5,Hit@6,Recall@6
0,Why is the attention dot product divided by th...,[transformer],"[transformer, transformer, transformer, dpr, t...",1.0,1,1.0,1,1.0,1,1.0,1,1.0
1,"How are token, segment, and position embedding...",[bert],"[bert, bert, colbert, colbert, bert, bert]",1.0,1,1.0,1,1.0,1,1.0,1,1.0
2,Which pooling strategies are considered when t...,[sbert],"[sbert, sbert, sbert, bert, sbert, sbert]",1.0,1,1.0,1,1.0,1,1.0,1,1.0
3,What kinds of negative passages are used to tr...,[dpr],"[dpr, dpr, dpr, dpr, dpr, dpr]",1.0,1,1.0,1,1.0,1,1.0,1,1.0
4,How does ColBERT's MaxSim operation use query-...,[colbert],"[colbert, colbert, colbert, colbert, colbert, ...",1.0,1,1.0,1,1.0,1,1.0,1,1.0
5,At what stage does Fusion-in-Decoder combine i...,[fid],"[fid, fid, fid, fid, fid, dpr]",1.0,1,1.0,1,1.0,1,1.0,1,1.0
6,How is the hypothetical document generated by ...,[hyde],"[hyde, hyde, hyde, hyde, hyde, rag]",1.0,1,1.0,1,1.0,1,1.0,1,1.0
7,How does question-answering performance change...,[lost_middle],"[lost_middle, lost_middle, lost_middle, lost_m...",1.0,1,1.0,1,1.0,1,1.0,1,1.0
8,How can Self-RAG decide whether retrieval is n...,[self_rag],"[self_rag, self_rag, self_rag, self_rag, self_...",1.0,1,1.0,1,1.0,1,1.0,1,1.0
9,What purpose do distractor documents serve dur...,[raft],"[raft, raft, raft, raft, raft, raft]",1.0,1,1.0,1,1.0,1,1.0,1,1.0


### Holdout results

The final 700-token collection is evaluated on **12 new queries** that were not used to choose the retriever or chunk size.

| Metric | Holdout |
|---|---:|
| MRR@6 | **1.0000** |
| Hit@1 | **1.0000** |
| Recall@1 | 0.9167 |
| Recall@3 | 0.9583 |
| Recall@5 | **0.9583** |
| Recall@6 | **0.9583** |

Every holdout query has at least one expected paper at rank 1. The remaining recall gap comes from one multi-source query: **DPR vs. ColBERT** retrieves ColBERT but not DPR within the final top 6.

That failure is useful because it shows the limitation of the source-level benchmark. `MRR@6 = 1.0` does not mean that every multi-source question has all required evidence. For multi-source RAG, coverage matters independently of first-hit ranking.

## 11. End-to-end grounded answers

In [40]:
# A mix of single-source and multi-source questions.
final_generation_indices = [0, 3, 7, 10, 11]

production_generation_results = []

for example_number, holdout_idx in enumerate(
    final_generation_indices,
    start=1,
):
    entry = final_holdout_benchmark[
        holdout_idx
    ]
    question = entry["query"]

    answer, sources, usage, cost = (
        production_answer(
            question=question,
        )
    )

    production_generation_results.append({
        "question": question,
        "expected_source_ids": entry[
            "expected_source_ids"
        ],
        "answer": answer,
        "sources": sources,
        "usage": usage,
        "cost": cost,
    })

    print("=" * 100)
    print(
        f"FINAL EXAMPLE {example_number} "
        f"| HOLDOUT INDEX {holdout_idx}"
    )
    print("=" * 100)

    print("\nQUESTION")
    print(question)

    print("\nEXPECTED PAPER SOURCES")
    print(entry["expected_source_ids"])

    print("\nANSWER")
    print(answer)

    print("\nFINAL CONTEXT SOURCES")
    for source in sources:
        print(
            f"[SOURCE {source['source']}] "
            f"paper_id={source['paper_id']} | "
            f"page={source['page']} | "
            f"paper_chunk_index="
            f"{source['paper_chunk_index']}"
        )

    print(
        f"\nEstimated API cost: "
        f"${cost:.6f}"
    )
    print()

FINAL EXAMPLE 1 | HOLDOUT INDEX 0

QUESTION
Why is the attention dot product divided by the square root of the key dimension?

EXPECTED PAPER SOURCES
['transformer']

ANSWER
The dot product is divided by \(\sqrt{d_k}\) to prevent attention scores from becoming excessively large as the key/query dimension \(d_k\) increases.

If the components of query \(q\) and key \(k\) are independent, have mean \(0\), and variance \(1\), then

\[
q\cdot k=\sum_{i=1}^{d_k}q_i k_i
\]

has variance \(d_k\). Thus, its typical magnitude grows roughly as \(\sqrt{d_k}\). Without scaling, large dot products push the softmax into a saturated region, where its gradients are extremely small and learning becomes difficult. Dividing by \(\sqrt{d_k}\) keeps the scores at a more stable scale and avoids this effect. [SOURCE 2]

The resulting attention computation is therefore

\[
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V.
\]

This scaling is particularly importa

### Qualitative generation analysis

The single-source examples are strongly grounded: the Transformer scaling question, DPR negative-sampling question, and *Lost in the Middle* question are answered from the expected paper with explicit source citations.

The two multi-source examples are more informative:

- **Self-RAG vs. RAFT** retrieves both expected papers and produces a grounded comparison that distinguishes inference-time adaptive retrieval from retrieval-aware fine-tuning.
- **DPR vs. ColBERT** exposes the final pipeline's main failure. The retrieved context contains ColBERT but not DPR. The generator explicitly says that DPR is not present by name and limits the comparison to the single-vector paradigm described in the supplied ColBERT evidence.

This is preferable to silently hallucinating missing evidence, but it also demonstrates that strong aggregate source-retrieval metrics do not guarantee complete evidence coverage for every comparison question.

## 12. Out-of-corpus abstention

In [41]:
with open(
    "evaluation/unanswerable_queries.json",
    encoding="utf-8",
) as f:
    unanswerable_queries = json.load(f)

unanswerable_queries

[{'query': 'How does LoRA decompose parameter updates into low-rank matrices?'},
 {'query': 'How does FlashAttention reduce memory traffic with IO-aware tiling?'},
 {'query': 'How does the selective state-space mechanism in Mamba work?'},
 {'query': 'What contrastive objective is used to train CLIP?'},
 {'query': 'What optimizer and exact training dataset were used for GPT-4?'}]

In [42]:
ABSTENTION_PATTERNS = (
    "does not provide enough evidence",
    "doesn't provide enough evidence",
    "not enough evidence",
    "insufficient evidence",
    "cannot answer",
    "can't answer",
    "cannot determine",
    "supplied context does not",
    "provided context does not",
    "corpus does not",
)


def detect_explicit_abstention(answer):
    normalized = answer.lower()
    return any(
        phrase in normalized
        for phrase in ABSTENTION_PATTERNS
    )


final_unanswerable_results = []

for i, entry in enumerate(
    unanswerable_queries,
    start=1,
):
    question = (
        entry
        if isinstance(entry, str)
        else entry["query"]
    )

    answer, sources, usage, cost = (
        production_answer(
            question=question,
        )
    )

    heuristic_abstained = (
        detect_explicit_abstention(
            answer
        )
    )

    final_unanswerable_results.append({
        "question": question,
        "answer": answer,
        "heuristic_abstained": heuristic_abstained,
        "cost": cost,
        "num_context_sources": len(sources),
    })

    print("=" * 100)
    print(f"OUT-OF-CORPUS QUESTION {i}")
    print("=" * 100)
    print("\nQuestion:")
    print(question)

    print("\nAnswer:")
    print(answer)

    print(
        "\nExplicit-abstention heuristic:",
        heuristic_abstained,
    )

    print(
        f"Estimated API cost: "
        f"${cost:.6f}"
    )
    print()


final_unanswerable_results_df = pd.DataFrame(
    final_unanswerable_results
)

heuristic_abstention_rate = (
    final_unanswerable_results_df[
        "heuristic_abstained"
    ].mean()
)

print(
    "Heuristic explicit-abstention rate:",
    f"{heuristic_abstention_rate:.3f}",
)

display(
    final_unanswerable_results_df[
        [
            "question",
            "heuristic_abstained",
            "cost",
            "num_context_sources",
        ]
    ]
)

OUT-OF-CORPUS QUESTION 1

Question:
How does LoRA decompose parameter updates into low-rank matrices?

Answer:
The supplied sources do not describe LoRA or its parameter-update decomposition. They only describe standard learned weight matrices in Transformer attention, such as \(W_i^Q\), \(W_i^K\), \(W_i^V\), and \(W^O\) [SOURCE 2]. Therefore, there is insufficient evidence here to explain how LoRA represents updates as low-rank matrices.

Explicit-abstention heuristic: True
Estimated API cost: $0.002038

OUT-OF-CORPUS QUESTION 2

Question:
How does FlashAttention reduce memory traffic with IO-aware tiling?

Answer:
The supplied sources only establish that FlashAttention is an **IO-aware CUDA kernel** that provides faster **exact attention**, making long-context training more efficient [SOURCE 8]. They do not describe the tiling procedure or quantify how it reduces memory traffic.

Thus, based strictly on the context, FlashAttention reduces memory traffic by using an IO-aware implement

,question,heuristic_abstained,cost,num_context_sources
0,How does LoRA decompose parameter updates into...,True,0.002038,16
1,How does FlashAttention reduce memory traffic ...,False,0.002359,18
2,How does the selective state-space mechanism i...,False,0.002140,17
3,What contrastive objective is used to train CLIP?,True,0.001814,14
4,What optimizer and exact training dataset were...,False,0.002089,16


### Manual abstention review

The phrase-based heuristic reports only `0.40`, but this is a limitation of the heuristic, not the RAG system.

Manual review of the five answers gives:

| Query | Manual outcome |
|---|---|
| LoRA decomposition | Correctly refuses to explain unsupported details |
| FlashAttention IO-aware tiling | Gives only the supported high-level statement, then explicitly refuses unsupported tiling details |
| Mamba selective state space | Correct abstention |
| CLIP objective | Correct abstention |
| GPT-4 optimizer / exact dataset | Correct abstention |

**Manual abstention result: 5 / 5.**

The heuristic misses three valid refusals because the wording differs from its small phrase list. For portfolio reporting, the manual result is used and the heuristic is retained only as an example of why semantic evaluation cannot be replaced by brittle keyword matching.

## 13. Cost

In [43]:
generation_cost = sum(
    result["cost"]
    for result in production_generation_results
)

abstention_cost = (
    final_unanswerable_results_df[
        "cost"
    ].sum()
)

total_final_api_cost = (
    generation_cost
    + abstention_cost
)

print("=" * 100)
print("RAG EXECUTION SUMMARY")
print("=" * 100)

print("\nOPERATIONAL CONFIGURATION")
print(
    "candidate_k =",
    OPERATIONAL_CANDIDATE_K,
)
print(
    "final_k =",
    OPERATIONAL_FINAL_K,
)
print(
    "neighbor_radius = 1"
)

print("\nSELECTED CHUNK SIZE")
print(optimal_chunk_size)

print("\nCHUNK-SIZE DEVELOPMENT ABLATION")
display(chunk_size_benchmark)

print("\nFINAL COLLECTION — DEVELOPMENT BENCHMARK")
display(production_dev_summary)

print("\nFINAL SOURCE-LEVEL HOLDOUT")
display(final_holdout_summary)

print("\nOUT-OF-CORPUS HEURISTIC")
print(
    "explicit abstention rate =",
    f"{heuristic_abstention_rate:.3f}",
)

print("\nAPI COST FOR FINAL QUALITATIVE CHECKS")
print(
    f"generation examples: "
    f"${generation_cost:.6f}"
)
print(
    f"out-of-corpus checks: "
    f"${abstention_cost:.6f}"
)
print(
    f"total: "
    f"${total_final_api_cost:.6f}"
)

RAG EXECUTION SUMMARY

OPERATIONAL CONFIGURATION
candidate_k = 20
final_k = 6
neighbor_radius = 1

SELECTED CHUNK SIZE
700

CHUNK-SIZE DEVELOPMENT ABLATION


,Chunk Size,Chunk Count,Hit@3,Recall@5,Recall@6,MRR@6
0,250,975,1.00,0.900,0.925,1.0000
1,450,500,0.95,0.925,0.925,0.9625
2,700,327,1.00,0.925,0.925,1.0000



FINAL COLLECTION — DEVELOPMENT BENCHMARK


RR@6        1.000
Hit@1       1.000
Recall@1    0.800
Hit@3       1.000
Recall@3    0.875
Hit@5       1.000
Recall@5    0.925
Hit@6       1.000
Recall@6    0.925
dtype: float64


FINAL SOURCE-LEVEL HOLDOUT


RR@6        1.000000
Hit@1       1.000000
Recall@1    0.916667
Hit@3       1.000000
Recall@3    0.958333
Hit@5       1.000000
Recall@5    0.958333
Hit@6       1.000000
Recall@6    0.958333
dtype: float64


OUT-OF-CORPUS HEURISTIC
explicit abstention rate = 0.400

API COST FOR FINAL QUALITATIVE CHECKS
generation examples: $0.010728
out-of-corpus checks: $0.010439
total: $0.021167


The final qualitative evaluation is inexpensive:

- five grounded answer examples: **$0.01086**
- five out-of-corpus checks: **$0.01069**
- total final API cost: **$0.02155**

Embedding, sparse retrieval, vector storage, fusion, and reranking run locally, so the hosted model is reserved for generation where it adds the most value.

## 14. Final conclusions

The experiments support the final architecture rather than selecting it only because it is more complex.

### Main findings

- Dense and BM25 retrieval have complementary failure modes.
- RRF materially improves **early-ranking robustness**, reaching `Hit@1 = 1.0` on the development benchmark.
- Hybrid fusion does not universally improve deep recall; some multi-source queries are better served by one component retriever.
- Cross-encoder reranking slightly worsens rank-1 metrics but raises `Recall@5` from **0.875 to 0.925**, so it is retained for evidence coverage.
- A 700-token chunk size matches or improves the selected retrieval metrics while reducing the index to **327 chunks**, compared with 975 at 250 tokens.
- The locked holdout achieves **MRR@6 = 1.0**, **Hit@1 = 1.0**, and **Recall@5 = 0.9583**.
- The remaining holdout failure is a multi-source DPR/ColBERT comparison where only one required paper survives the final cutoff.
- The generator behaves appropriately when evidence is incomplete: it acknowledges the missing DPR evidence instead of presenting unsupported claims as sourced facts.
- All five deliberately out-of-corpus questions are manually judged to abstain correctly.
- The final hosted-generation checks cost only about **$0.02**.

### What the project demonstrates

The main engineering lesson is that RAG quality cannot be reduced to a single generation metric.

A failure can occur at different stages:

```text
relevant source absent from candidates
        → retrieval recall

relevant source in candidates but removed by ranking
        → ranking / cutoff

correct paper retrieved but wrong chunk
        → evidence granularity

sufficient evidence supplied but unsupported answer generated
        → grounding / generation
```

Evaluating retrieval independently makes those failure modes observable instead of attributing every bad answer to the LLM.

## 15. Limitations

1. **Small, curated corpus.** The 11-paper collection is useful for controlled evaluation but does not stress Qdrant indexing or latency at production scale.

2. **Source-level relevance labels.** The benchmark labels expected papers, not exact answer-bearing passages. A correct `paper_id` can therefore count as relevant even when the retrieved chunk is not sufficient to answer the question.

3. **Small benchmark.** Twenty development questions and twelve holdout questions are enough to compare design choices, but the resulting metrics have high sampling uncertainty.

4. **Multi-source coverage remains the main retrieval weakness.** Multiple chunks from one relevant paper can dominate the top results and crowd out a second required source.

5. **No automated semantic answer judge.** Generation quality is inspected manually rather than through Ragas or an LLM judge. This avoids pretending that an evaluator model is ground truth, but it also limits scale.

6. **PDF extraction is text-first.** Figures, equations, tables, and layout-heavy content are not reconstructed semantically, so questions whose answer depends on those elements may be harder.

7. **Local models favor laptop-scale reproducibility over maximum retrieval quality.** Larger embedding or reranking models could improve quality at higher compute or API cost.

## 16. Final architecture

```text
11 arXiv papers
      ↓
PyMuPDF + source/page metadata
      ↓
700-token chunks / 75-token overlap
      ↓
┌────────────────────────────┐
│ BGE-small dense retrieval  │
│ Qdrant BM25 sparse search  │
└─────────────┬──────────────┘
              ↓
          Qdrant RRF
       20 candidates
              ↓
 MiniLM cross-encoder reranker
          top 6
              ↓
     ±1 context expansion
              ↓
       GPT-5.6 Luna
              ↓
 grounded answer + citations
```

The final design keeps retrieval and reranking local and uses a hosted LLM only for grounded generation.